In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from plotly import express as px
from plotly import graph_objects as go

In [ ]:
df=pd.read_csv('gandhinagar_property_feature_selection.csv')
df.head()

,Area_sqft,bathrooms,balconies,current_floor,total_floors,furnishing_status,Mapped_Area,facing,property_age_bucket,is_ready_to_move,is_built_up_area,is_carpet_area,is_super_built_up_area,Bedrooms,price
0,2916.0,3.0,1.0,4.0,8.0,0.0,19.0,8.0,3.0,1,False,False,True,3.0,126.00
1,1755.0,3.0,2.0,3.0,7.0,0.0,13.0,8.0,3.0,0,False,False,True,3.0,51.99
2,1908.0,3.0,2.0,4.0,8.0,0.0,16.0,8.0,3.0,1,False,False,True,3.0,97.00
3,2205.0,2.0,2.0,9.0,13.0,0.0,16.0,8.0,3.0,1,False,True,False,3.0,125.00
4,1755.0,3.0,1.0,8.0,13.0,3.0,14.0,8.0,2.0,1,False,True,False,3.0,79.00


In [52]:
X=df.drop(columns=['price'])
y=df['price']

In [53]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

In [54]:
columns_to_encode=['furnishing_status','Mapped_Area','facing','property_age_bucket']

In [55]:
y_transformed = np.log1p(y)

In [56]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['Area_sqft', 'Bedrooms', 'balconies','bathrooms', 'current_floor', 'total_floors', 'is_ready_to_move','is_carpet_area','is_super_built_up_area','is_built_up_area']),
        ('cat', OneHotEncoder(drop='first',handle_unknown='ignore'), columns_to_encode)
    ],
    remainder='passthrough'
)

In [57]:
pipeline=Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', SVR(kernel='rbf'))
    # ('regressor',LinearRegression())
])

In [58]:
kflod=KFold(n_splits=10, shuffle=True, random_state=42)
scores=cross_val_score(pipeline, X, y_transformed, cv=kflod, scoring='r2')

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders

In [59]:
scores.mean()

np.float64(0.8432584113153336)

In [60]:
scores.std()

np.float64(0.038328801865212106)

In [61]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)

In [62]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

c:\Users\MEGH BAVARVA\Desktop\Data Scientist\Project\Real_Estate\Code\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [63]:
y_pred_exp = np.expm1(y_pred)

In [64]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(np.expm1(y_test), y_pred_exp)
print(f'Mean Absolute Error: {mae}')

Mean Absolute Error: 16.732300048627124
